<a href="https://colab.research.google.com/github/henry09027/NTU-Course-MachineLearning/blob/main/Homework/HW1/Code/2026ML_HW1_V6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **2026 ML FALL HW1: PM2.5 Prediction (Regression)**

Author: MLTAs

Methods:
* Training with all data
* Optimizer: RMSProp (default)
* TODOs:
  - Complete the `valid()` function
  - Tune the hyperparameters in `train_config`
  - Implement 2nd-order polynomial regression model (without interaction terms) in `minibatch_2()`
  - Implement feature normalization in `normalize_train_data()`
  - Feature selection



# **Import Some Packages**

In [1]:
import numpy as np
import csv
import math
import pandas as pd
import os

# **Fix random seed**


This is for the reproduction of your result. **DO NOT modify this secton!**


In [2]:
seed = 9487
np.random.seed(seed)

# **Download training data**


In [3]:
!gdown --id "1Hfzrcm69QwdFvdeF0uASoQlcVxKw_hHy" --output "train.csv"
!gdown --id '155N6fzI7vAFzHAGdy6jkaWIksWH6Y1G2' --output "test.csv"

# Incase the links above die, you can use the following instead.
#!gdown --id '11abE854Eyv4BA7qt5k8r_80sJ3KuOQUN' --output "train.csv"
#!gdown --id '1uod-Z4ztluXnuHtgUbm39nMudUKqXHMl' --output "test.csv"

# If the data is still missing, you can manually download it from kaggle, and upload the files under /content

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Hfzrcm69QwdFvdeF0uASoQlcVxKw_hHy
To: /content/train.csv
100% 324k/324k [00:00<00:00, 26.2MB/s]
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=155N6fzI7vAFzHAGdy6jkaWIksWH6Y1G2
To: /content/test.csv
100% 49.0k/49.0k [00:00<00:00, 46.9MB/s]


In [4]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(train_df['PM2.5'].corr(train_df['PM10']))
print(train_df['PM2.5'].mean(), test_df['PM2.5'].mean())
display(train_df[train_df['PM2.5'] > 30])
display(test_df[test_df['PM2.5'] >= 30])
display(train_df)

0.8188682137702157
11.582784897817804 7.836111111111111


,AMB_TEMP,CO,NO,NO2,NOx,O3,PM10,WS_HR,RAINFALL,RH,SO2,WD_HR,WIND_DIREC,WIND_SPEED,PM2.5
265,11.6,0.50,4.0,15.7,19.8,31.6,61,0.3,0.0,63,4.0,305,169,0.7,34
266,12.5,0.55,6.3,17.5,23.8,34.7,55,0.5,0.0,60,3.9,192,15,0.8,36
267,12.9,0.46,3.6,15.1,18.7,42.9,60,0.7,0.0,56,3.6,11,74,1.1,35
358,18.2,0.51,1.9,11.0,13.0,53.0,67,1.7,0.0,72,1.5,26,12,2.0,39
359,17.1,0.51,1.4,13.2,14.7,50.5,76,1.4,0.0,76,1.5,34,22,2.2,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2819,25.0,0.69,4.1,36.8,41.0,12.9,82,0.5,0.0,79,3.6,42,60,0.9,38
2821,24.7,0.83,10.2,37.9,48.1,1.9,70,0.7,0.0,81,3.3,171,176,1.2,34
2822,24.4,0.74,8.0,28.1,36.2,3.1,56,1.0,0.0,83,3.2,182,155,0.5,32
2996,32.5,0.55,2.1,15.4,17.5,94.3,52,1.9,0.0,59,3.5,340,358,3.4,35


,AMB_TEMP,CO,NO,NO2,NOx,O3,PM10,WS_HR,RAINFALL,RH,SO2,WD_HR,WIND_DIREC,WIND_SPEED,PM2.5
670,18.9,0.34,0.9,11.8,12.7,55.2,55.0,1.0,0.0,74.0,1.1,93.0,114.0,1.8,30.0


,AMB_TEMP,CO,NO,NO2,NOx,O3,PM10,WS_HR,RAINFALL,RH,SO2,WD_HR,WIND_DIREC,WIND_SPEED,PM2.5
0,10.8,0.32,1.7,8.6,10.3,22.9,21,0.6,0.0,71,1.9,172,171,0.6,15
1,10.8,0.27,1.6,6.2,7.8,23.8,20,1.4,0.0,71,1.7,161,129,1.8,13
2,11.0,0.25,0.9,5.4,6.3,27.4,21,0.8,0.0,68,1.6,152,147,1.5,12
3,11.0,0.23,0.7,3.1,3.8,29.5,21,1.8,0.0,68,1.6,138,145,1.7,9
4,11.3,0.22,0.8,2.9,3.8,30.7,16,1.9,0.0,67,1.6,140,139,1.7,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5769,29.0,0.41,1.2,14.0,15.3,23.0,21,1.4,0.0,74,2.8,149,168,2.0,14
5770,28.2,0.33,1.7,11.7,13.5,19.5,23,2.1,0.0,78,2.3,187,179,2.5,15
5771,28.0,0.29,1.3,9.1,10.4,17.6,17,1.5,0.0,78,2.0,173,200,1.5,13
5772,28.0,0.27,1.4,9.5,11.0,15.4,17,1.1,0.0,75,1.8,171,135,0.9,10


In [5]:
def valid(x, y):
  # Filter out extreme values. In test.csv every PM2.5 value (inputs) is <= 30, so training windows containing
  # PM2.5 spikes (train goes up to 80) come from a different regime and only distort the fit.
  # Keep a window only if the target and all 8 input PM2.5 hours are <= 30.
  # (feats contains only PM2.5, which is NOT normalized, so x.max() / y are in raw PM2.5 units here.)
  return y <= 30 and x.max() <= 30


# Create your dataset
def parse2train(data, feats):

  x = []
  y = []

  # Use data #0~#7 to predict #8 => Total data length should be decresased by 8.
  total_length = data.shape[1] - 8

  for i in range(total_length):
    x_tmp = data[feats, i:i+8] # Use data #0~#7 to predict #8, data #1~#8 to predict #9, etc.
    y_tmp = data[-1, i+8] # last column of (i+8)th row: PM2.5

    # Filter out extreme values to train.
    if valid(x_tmp, y_tmp):
      x.append(x_tmp.reshape(-1,))
      y.append(y_tmp)

  # x.shape: (n, 15, 8)
  # y.shape: (n, 1)
  x = np.array(x)
  y = np.array(y)

  return x,y


#**Gradient descent**
###**RMSProp**
1. $v_t=\beta \cdot v_{t-1} + (1-\beta)(\nabla w_t)^2$
2. $w_{t+1}=w_t - \frac{\eta}{\sqrt{(v_t)}+\epsilon}\nabla w_t$




* This is our gradient descent algorithm. RMSProp was implemented in `minibatch()`.
* You can implement other algorithm, such as SGD or other gradient descent variants listed below, which may (or may not) improve performance.
* However, **modules like sklearn and pytorch are not allowed!!!**
* Ref:
  - Prof. G. Hinton's lecture: https://www.cs.toronto.edu/~tijmen/csc321/slides/lecture_slides_lec6.pdf
  - Prof. Hung-Yi Lee's video: https://youtu.be/HYUXEeh3kwY?si=RtLjSj51WK1pmz87

###**Adam (RMSProp + Momemtum)**
* Ref:
  - Paper: https://arxiv.org/pdf/1412.6980
  - Prof. Hung-Yi Lee's video: https://youtu.be/HYUXEeh3kwY?si=RtLjSj51WK1pmz87

###**AdamW (Adam with decoupled weight decay)**
* Ref:
  - Paper: https://arxiv.org/pdf/1711.05101




In [6]:
def minibatch(x, y, config):
    # Randomize the data in minibatch
    index = np.arange(x.shape[0])
    np.random.shuffle(index)
    x = x[index]
    y = y[index]

    # Initialization
    batch_size = config.batch_size
    lr = config.lr
    epoch = config.epoch
    decay_rate = config.decay_rate
    epsilon = 1e-8

    # Linear regression: only contains two parameters (w, b).
    w = np.full(x[0].shape, 0.1).reshape(-1, 1)
    bias = 0.1

    # Optimizer states
    cache_w = np.zeros_like(w)
    cache_b = 0.0

    # Training loop
    for num in range(epoch):
        for b in range(int(x.shape[0] / batch_size)):
            x_batch = x[b * batch_size:(b + 1) * batch_size]
            y_batch = y[b * batch_size:(b + 1) * batch_size].reshape(-1, 1)

            # Prediction of linear regression
            pred = np.dot(x_batch, w) + bias

            # Loss
            loss = y_batch - pred

            # Compute gradient
            g_t = np.dot(x_batch.transpose(), loss) * (-2)
            g_t_b = loss.sum(axis=0) * (-2)

            # Update cache
            cache_w = decay_rate * cache_w + (1 - decay_rate) * g_t**2
            cache_b = decay_rate * cache_b + (1 - decay_rate) * g_t_b**2

            # Update weight & bias
            w -= lr * g_t / (np.sqrt(cache_w) + epsilon)
            bias -= lr * g_t_b / (np.sqrt(cache_b) + epsilon)

    return w, bias

# 2nd-order polynomial regression (no interaction terms): y = b + sum_i (w1_i * x_i + w2_i * x_i^2)
# Each feature is expanded to [x, x^2], then trained with the same RMSProp routine as minibatch().
# Returned w has shape (2 * n_features, 1): the first half are the linear terms, the second half the squared terms.
# To predict, expand the test input the same way: np.concatenate([test_x, test_x**2], axis=1).
def minibatch_2(x, y, config):
    x2 = np.concatenate([x, x ** 2], axis=1)
    return minibatch(x2, y, config)


In [7]:
from argparse import Namespace

# TODO: Tune the config to boost your performance.
# RMSProp with a constant lr keeps jittering around the optimum, so a small lr + larger batch + many epochs is used to land
# very close to the closed-form least-squares solution (checked numerically).
train_config = Namespace(
    batch_size = 64,
    lr = 0.0003,
    epoch = 300,
    decay_rate = 0.9
)

# The 2nd-order model has squared inputs (e.g. PM2.5^2 up to ~1600), so it needs a smaller learning rate to stay stable.
train_config_2 = Namespace(
    batch_size = 32,
    lr = 0.0003,
    epoch = 100,
    decay_rate = 0.9
)

# **Training your regression model**

In [8]:
train_df = pd.read_csv("/content/train.csv")
train_df

,AMB_TEMP,CO,NO,NO2,NOx,O3,PM10,WS_HR,RAINFALL,RH,SO2,WD_HR,WIND_DIREC,WIND_SPEED,PM2.5
0,10.8,0.32,1.7,8.6,10.3,22.9,21,0.6,0.0,71,1.9,172,171,0.6,15
1,10.8,0.27,1.6,6.2,7.8,23.8,20,1.4,0.0,71,1.7,161,129,1.8,13
2,11.0,0.25,0.9,5.4,6.3,27.4,21,0.8,0.0,68,1.6,152,147,1.5,12
3,11.0,0.23,0.7,3.1,3.8,29.5,21,1.8,0.0,68,1.6,138,145,1.7,9
4,11.3,0.22,0.8,2.9,3.8,30.7,16,1.9,0.0,67,1.6,140,139,1.7,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5769,29.0,0.41,1.2,14.0,15.3,23.0,21,1.4,0.0,74,2.8,149,168,2.0,14
5770,28.2,0.33,1.7,11.7,13.5,19.5,23,2.1,0.0,78,2.3,187,179,2.5,15
5771,28.0,0.29,1.3,9.1,10.4,17.6,17,1.5,0.0,78,2.0,173,200,1.5,13
5772,28.0,0.27,1.4,9.5,11.0,15.4,17,1.1,0.0,75,1.8,171,135,0.9,10


In [9]:
# TODO: Normalize each column (except PM2.5) for the report (use z-score normalization)
def normalize_train_data(df):
    """
    Steps:
    1. For each column (except PM2.5): calculate mean and std
    2. Apply standardization: (column - mean) / std
    3. Store normalization parameters for later use on test data

    Returns:
        normalized_df: DataFrame with normalized features
        norm_params: Dict with {'column': {'mean': X, 'std': Y}}

    Hint: Loop through data.columns, skip PM2.5
    """
    data_norm = df.copy()
    norm_params = {}

    for col in df.columns:
        if col == 'PM2.5':  # keep PM2.5 in its raw unit (it is also the prediction target)
            continue
        mean = df[col].mean()
        std = df[col].std()
        data_norm[col] = (df[col] - mean) / std
        norm_params[col] = {'mean': mean, 'std': std}

    return data_norm, norm_params

In [10]:
# Choose your features to train.
# Hint:
# 1. You can select more than one feature.
# 2. You should select "good" features.

# TODO: Carefully justify which feature should be chosen.
# Index: 0 AMB_TEMP, 1 CO, 2 NO, 3 NO2, 4 NOx, 5 O3, 6 PM10, 7 WS_HR, 8 RAINFALL, 9 RH, 10 SO2,
#        11 WD_HR, 12 WIND_DIREC, 13 WIND_SPEED, 14 PM2.5
# Selected: PM10 and PM2.5 (6, 14) only.
# Justification:
# - The test set comes from a cleaner, noisier-at-hour-level regime than train (mean PM2.5 7.8 vs 11.6, SO2 0.9 vs 1.7, RH 81 vs 73,
#   lag-1 autocorrelation of PM2.5 0.60 vs 0.85).
# - Wind-direction columns are circular (0 and 359 degrees are neighbours), so a linear model cannot use them anyway.

#feats = [14]
feats = [6, 14]

In [11]:
# Training data preprocessing
def train_processing(train_df, norm=False):
    """Process training train_df with optional normalization"""

    if norm:
        # Normalize training data and save parameters (mean & std)
        data_norm, norm_params = normalize_train_data(train_df)
        data_values = data_norm.values
    else:
        # Use raw training data
        data_values = train_df.values
        norm_params = None

    # Common processing steps
    train_data = np.transpose(np.array(np.float64(data_values)))
    train_x, train_y = parse2train(train_data, feats)

    return train_x, train_y, norm_params

train_x, train_y, norm_params = train_processing(train_df, norm=True)

In [12]:
# Train your regression model
w, bias = minibatch(train_x, train_y, train_config)
print(w.shape, bias.shape)

(16, 1) (1,)


# **Testing:**


In [13]:
def parse2test(data, feats):
  x = []
  for i in range(90):
    x_tmp = data[feats,8*i: 8*i+8]
    x.append(x_tmp.reshape(-1,))

  # x.shape: (n, 15, 8)
  x = np.array(x)
  return x

In [14]:
def normalize_test_data(df, norm_params):
    data_norm = df.copy()

    for col, params in norm_params.items():
        if col in df.columns:
            data_norm[col] = (df[col] - params['mean']) / params['std']

    return data_norm

In [15]:
test_df = pd.read_csv('test.csv')
test_df

,AMB_TEMP,CO,NO,NO2,NOx,O3,PM10,WS_HR,RAINFALL,RH,SO2,WD_HR,WIND_DIREC,WIND_SPEED,PM2.5
0,27.5,0.22,0.7,9.0,9.8,13.2,31.0,1.2,0.0,79.0,1.7,180.0,171.0,1.2,20.0
1,27.2,0.17,0.4,5.0,5.4,15.7,20.0,1.5,0.0,79.0,1.6,192.0,187.0,1.9,8.0
2,26.8,0.17,0.4,4.3,4.8,12.8,16.0,1.6,0.0,81.0,1.3,181.0,180.0,1.8,9.0
3,26.7,0.19,0.4,4.1,4.5,12.0,21.0,1.7,0.0,80.0,1.5,179.0,188.0,2.3,6.0
4,26.4,0.22,0.4,4.1,4.6,10.1,23.0,2.2,0.0,81.0,1.5,184.0,186.0,1.9,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,16.0,0.26,0.3,3.9,4.2,47.1,34.0,2.7,0.0,70.0,0.5,130.0,133.0,2.8,13.0
716,15.6,0.25,0.4,3.3,3.7,44.1,27.0,3.0,0.0,74.0,0.6,136.0,131.0,3.2,15.0
717,15.7,0.24,0.4,3.7,4.1,44.1,29.0,2.9,0.0,73.0,0.6,133.0,129.0,2.7,12.0
718,15.1,0.24,0.6,10.5,11.1,29.9,9.0,0.8,0.0,95.0,0.6,24.0,21.0,1.1,8.0


In [16]:
# Testing data preprocessing
def test_processing(test_df, norm=False, norm_params=norm_params):
    if norm:
        if norm_params is None:
            raise ValueError("norm_params required when norm=True")

        # Apply training normalization parameters to testing data
        data_norm = normalize_test_data(test_df, norm_params)
        data_values = data_norm.values
    else:
        # Use raw testing data
        data_values = test_df.values

    # Common processing steps
    test_data = np.transpose(np.array(np.float64(data_values)))
    test_x = parse2test(test_data, feats)

    return test_x

test_x = test_processing(test_df, norm=True, norm_params=norm_params)

# **Write Main result as .csv**

---



In [17]:
 with open('my_sol.csv', 'w', newline='') as csvf:
    writer = csv.writer(csvf)
    writer.writerow(['Id','Predicted'])

    print(test_x.shape)
    for i in range(int(test_x.shape[0])):
      # Prediction of linear regression
      prediction = (np.dot(np.reshape(w,-1),test_x[i]) + bias)[0]
      writer.writerow([i, prediction])

(90, 16)


In [18]:
my_sol_df = pd.read_csv('my_sol.csv')
display(my_sol_df)

,Id,Predicted
0,0,7.513188
1,1,2.119424
2,2,11.916177
3,3,13.664457
4,4,6.702316
...,...,...
85,85,5.465672
86,86,3.511231
87,87,6.343901
88,88,24.397891


In [19]:
from google.colab import files
files.download('my_sol.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Extra Kaggle experiments for the report (Q1 / Q2 / Q3)**

Everything above this line is the main pipeline. Run it first (it defines `train_df`, `test_df`, `valid`, `minibatch`, `minibatch_2`, `normalize_*`, `parse2test`, `train_config`).

The cells below do **not** modify any global (`feats`, `valid`, ...). Each experiment is one call of `run_experiment()`, which reseeds (9487), trains, and writes a csv in the same format as the main section (`Id,Predicted`).

Upload priority is printed in the manifest at the bottom.

In [20]:
# ============================================================
# Helpers for the extra experiments (nothing here touches the main section)
# ============================================================
import os, csv
import numpy as np
import pandas as pd
from argparse import Namespace

Q_SEED = 9487
Q_LOG = {}          # tag -> manifest row (re-running a cell overwrites its row)

# --- valid() variants ---------------------------------------------------------
# The main valid() uses x.max(), which scans EVERY selected feature. That is fine when the other
# features are z-scored (main run), but with norm=False it would also scan raw WD_HR (0-359) / raw PM10
# and silently throw away almost every training window. valid_pm25 applies the identical rule
# (target <= 30 and all 8 input PM2.5 <= 30) to the PM2.5 row only (always the LAST row of feats).
def valid_pm25(x, y):
    return y <= 30 and x[-1].max() <= 30

def valid_none(x, y):          # "no filtering" ablation
    return True

# --- configs --------------------------------------------------------------------
cfg_A = train_config                                                    # final-notebook config (batch 64, lr 3e-4, 300 ep)
cfg_B = Namespace(batch_size=32, lr=0.003, epoch=100, decay_rate=0.9)   # report config B (10x larger lr); decay_rate assumed 0.9
print('cfg_A =', cfg_A)
print('cfg_B =', cfg_B)

# --- parameterised copy of parse2train (feats / valid passed in, not read from globals) ---
def q_parse2train(data, feats_, valid_fn):
    x, y = [], []
    for i in range(data.shape[1] - 8):
        x_tmp = data[feats_, i:i+8]
        y_tmp = data[-1, i+8]                  # PM2.5 is the last column
        if valid_fn(x_tmp, y_tmp):
            x.append(x_tmp.reshape(-1,))
            y.append(y_tmp)
    return np.array(x), np.array(y)

def run_experiment(tag, feats_, norm, cfg, valid_fn, poly=False, save=True):
    """Train -> predict the 90 test rows -> write <tag>.csv (same format as the main section)."""
    # 1) preprocessing (re-uses the notebook's own normalisation functions)
    if norm:
        tr_df, params = normalize_train_data(train_df)
        te_df = normalize_test_data(test_df, params)
    else:
        tr_df, te_df = train_df, test_df
    tr = np.transpose(np.array(np.float64(tr_df.values)))
    te = np.transpose(np.array(np.float64(te_df.values)))
    train_x, train_y = q_parse2train(tr, feats_, valid_fn)
    test_x = parse2test(te, feats_)

    # 2) train (reseed so every experiment is independently reproducible)
    np.random.seed(Q_SEED)
    w, bias = (minibatch_2 if poly else minibatch)(train_x, train_y, cfg)
    w = np.reshape(w, -1)
    b = float(np.ravel(bias)[0])

    # 3) predict
    expand = (lambda a: np.concatenate([a, a ** 2], axis=1)) if poly else (lambda a: a)
    pred = expand(test_x) @ w + b
    train_rmse = float(np.sqrt(np.mean((expand(train_x) @ w + b - train_y) ** 2)))
    finite = bool(np.isfinite(pred).all())

    # 4) csv, identical layout to the main section
    fn = f'{tag}.csv'
    if save:
        with open(fn, 'w', newline='') as csvf:
            writer = csv.writer(csvf)
            writer.writerow(['Id', 'Predicted'])
            for i in range(len(pred)):
                writer.writerow([i, pred[i]])
        Q_LOG[tag] = dict(file=fn,
                          features=','.join(train_df.columns[j] for j in feats_),
                          norm=norm, valid=valid_fn.__name__, poly=poly,
                          lr=cfg.lr, batch=cfg.batch_size, epochs=cfg.epoch,
                          n_train=len(train_y), train_rmse=round(train_rmse, 4),
                          pred_mean=round(float(np.nanmean(pred)), 3), pred_std=round(float(np.nanstd(pred)), 3),
                          finite=finite)
    print(f"{tag:34s} n_train={len(train_y):5d}  train_RMSE={train_rmse:9.4f}  "
          f"pred mean/std={np.nanmean(pred):8.3f}/{np.nanstd(pred):8.3f}  {'saved ' + fn if save else '(not saved)'}")
    if not finite:
        print('   !! non-finite predictions (training diverged) - Kaggle will reject this csv. '
              'Report it as "diverged" or lower the lr; do not clip silently.')
    return dict(pred=pred, w=w, b=b, train_rmse=train_rmse, n_train=len(train_y))

cfg_A = Namespace(batch_size=64, lr=0.0003, epoch=300, decay_rate=0.9)
cfg_B = Namespace(batch_size=32, lr=0.003, epoch=100, decay_rate=0.9)


## **Q1 - data preprocessing ablation**
All rows: `norm=True` (z-score), `train_config`, seed 9487, only the stated factor changes.

| report row | features | valid() | status |
|---|---|---|---|
| 1 | CO,O3,PM10,RH,WIND_SPEED,PM2.5 | target > 40 dropped | already scored (3.53749) |
| 2 | PM2.5 | <= 30 | already scored (3.13460) |
| 3 | PM10,PM2.5 | <= 30 | = main result (3.05207) |
| **4** | PM10,PM2.5 | **none** | `q1_row4_...csv` **(required)** |
| **1b** (new) | same 6 features as row 1 | <= 30 | `q1_row1b_...csv` (recommended: removes the feature-vs-filter confound in row 1) |
| **5** (new) | PM10,PM2.5 | <= 30, **no z-score** | `q1_row5_...csv` (optional: direct evidence for normalisation on the final model) |

In [21]:
# ---------------- Q1 ----------------
RUN_OPTIONAL = True     # set False to skip the optional rows 1b / 5

# Sanity check: does this helper pipeline reproduce the main result? (nothing is saved)
chk = run_experiment('q1_check_row3', [6, 14], True, train_config, valid_pm25, save=False)
if os.path.exists('my_sol.csv'):
    ref = pd.read_csv('my_sol.csv')['Predicted'].values
    print('   max |pred - my_sol.csv| =', np.abs(ref - chk['pred']).max(), ' (~0 => helper pipeline == main pipeline)')

# Row 4 (REQUIRED): PM10 + PM2.5, normalised, NO filtering
run_experiment('q1_row4_PM10_PM25_nofilter', [6, 14], True, train_config, valid_none)

if RUN_OPTIONAL:
    # Row 1b: the 6 features of row 1 but with the same <=30 filter as rows 2-3  (vs row 3 => effect of features only)
    run_experiment('q1_row1b_6feats_filter30', [1, 5, 6, 9, 13, 14], True, train_config, valid_pm25)
    # Row 5: final model, same <=30 filter, but NO z-score         (vs row 3 => effect of normalisation only)
    run_experiment('q1_row5_PM10_PM25_nonorm', [6, 14], False, train_config, valid_pm25)

q1_check_row3                      n_train= 5170  train_RMSE=   3.0380  pred mean/std=   7.987/   3.905  (not saved)
   max |pred - my_sol.csv| = 5.329070518200751e-15  (~0 => helper pipeline == main pipeline)
q1_row4_PM10_PM25_nofilter         n_train= 5766  train_RMSE=   3.6211  pred mean/std=   8.065/   3.960  saved q1_row4_PM10_PM25_nofilter.csv
q1_row1b_6feats_filter30           n_train= 5170  train_RMSE=   2.9379  pred mean/std=   8.182/   3.913  saved q1_row1b_6feats_filter30.csv
q1_row5_PM10_PM25_nonorm           n_train= 5170  train_RMSE=   3.0358  pred mean/std=   8.045/   3.920  saved q1_row5_PM10_PM25_nonorm.csv


## **Q2 - 2nd-order polynomial regression, NO only**
`feats=[2]`, z-score, `minibatch_2`, the notebook's original `valid()` (target <= 30), seed 9487.
The cell prints the learned parameters next to the table already written in the report. If they match, the Kaggle score you get is the score of exactly those parameters.

In [22]:
# ---------------- Q2 ----------------
Q2_CFG = train_config       # the report says train_config; switch to train_config_2 ONLY if the check below does not match

q2 = run_experiment('q2_poly2_NO_only', [2], True, Q2_CFG, valid, poly=True)

# Parameter table (first 8 = x terms, last 8 = x^2 terms; hour 0 = oldest, 7 = most recent)
rep_w1 = np.array([-0.0639, -0.0466, 0.0430, 0.2173, 0.1358, -0.3408, -0.1883, -0.6058])
rep_w2 = np.array([ 0.0422,  0.0682, 0.0829, -0.0036, 0.0273,  0.1042,  0.0643,  0.1587])
rep_b  = 7.5856
q2_tab = pd.DataFrame({'hour': range(8),
                       'w1 (x)': q2['w'][:8],  'w1 in report': rep_w1,
                       'w2 (x^2)': q2['w'][8:], 'w2 in report': rep_w2})
display(q2_tab.round(4))
print('bias =', round(q2['b'], 4), ' (report:', rep_b, ')')
dmax = max(np.abs(q2['w'][:8] - rep_w1).max(), np.abs(q2['w'][8:] - rep_w2).max(), abs(q2['b'] - rep_b))
print('max abs diff vs report table:', round(float(dmax), 5))
print('MATCHES the report table -> upload q2_poly2_NO_only.csv' if dmax < 6e-4 else
      'DOES NOT MATCH -> either replace the report table with the values above, or change Q2_CFG / valid and rerun')

q2_poly2_NO_only                   n_train= 5503  train_RMSE=   6.9885  pred mean/std=   7.932/   1.146  saved q2_poly2_NO_only.csv


,hour,w1 (x),w1 in report,w2 (x^2),w2 in report
0,0,-0.0639,-0.0639,0.0422,0.0422
1,1,-0.0466,-0.0466,0.0682,0.0682
2,2,0.0430,0.0430,0.0829,0.0829
3,3,0.2173,0.2173,-0.0036,-0.0036
4,4,0.1358,0.1358,0.0273,0.0273
5,5,-0.3408,-0.3408,0.1042,0.1042
6,6,-0.1883,-0.1883,0.0643,0.0643
7,7,-0.6058,-0.6058,0.1587,0.1587


bias = 7.5856  (report: 7.5856 )
max abs diff vs report table: 5e-05
MATCHES the report table -> upload q2_poly2_NO_only.csv


## **Q3(b) - normalisation on / off, WD_HR + PM2.5**
`feats=[11, 14]`. Within each train_config, **only `norm` differs** (same seed, same filter, same config).
`valid_pm25` is required here: the main `valid()` would see raw WD_HR (0-359) when `norm=False` and discard nearly every window.
Config C (TA sample config) is local-only in the report, so no csv is produced for it.

In [23]:
# ---------------- Q3 ----------------
feats_wd = [11, 14]     # WD_HR, PM2.5
for cfg_name, cfg in [('A', cfg_A), ('B', cfg_B)]:
    for norm in (True, False):
        run_experiment(f"q3_cfg{cfg_name}_{'norm' if norm else 'nonorm'}", feats_wd, norm, cfg, valid_pm25)
# report (config A) quotes training RMSE 3.089 (norm) vs 3.096 (no norm) -> compare with train_RMSE above

q3_cfgA_norm                       n_train= 5170  train_RMSE=   3.0893  pred mean/std=   7.897/   3.733  saved q3_cfgA_norm.csv
q3_cfgA_nonorm                     n_train= 5170  train_RMSE=   3.0956  pred mean/std=   7.965/   3.721  saved q3_cfgA_nonorm.csv
q3_cfgB_norm                       n_train= 5170  train_RMSE=   3.0914  pred mean/std=   7.918/   3.756  saved q3_cfgB_norm.csv
q3_cfgB_nonorm                     n_train= 5170  train_RMSE=   3.3172  pred mean/std=   6.891/   3.721  saved q3_cfgB_nonorm.csv


## **Manifest & download**
`n_train` per row shows how many windows survived `valid()`; `finite=False` means that run diverged.

In [24]:
order = ['q1_row4_PM10_PM25_nofilter', 'q2_poly2_NO_only',
         'q3_cfgA_norm', 'q3_cfgA_nonorm', 'q3_cfgB_norm', 'q3_cfgB_nonorm',      # required (blank cells in the report)
         'q1_row1b_6feats_filter30', 'q1_row5_PM10_PM25_nonorm']                  # optional / recommended
log_df = pd.DataFrame([Q_LOG[t] for t in order if t in Q_LOG])
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
display(log_df)

try:
    from google.colab import files
    for f in log_df['file']:
        files.download(f)
except ImportError:
    print('Not on Colab - csv files are in', os.getcwd())

,file,features,norm,valid,poly,lr,batch,epochs,n_train,train_rmse,pred_mean,pred_std,finite
0,q1_row4_PM10_PM25_nofilter.csv,"PM10,PM2.5",True,valid_none,False,0.0003,64,300,5766,3.6211,8.065,3.960,True
1,q2_poly2_NO_only.csv,NO,True,valid,True,0.0003,64,300,5503,6.9885,7.932,1.146,True
2,q3_cfgA_norm.csv,"WD_HR,PM2.5",True,valid_pm25,False,0.0003,64,300,5170,3.0893,7.897,3.733,True
3,q3_cfgA_nonorm.csv,"WD_HR,PM2.5",False,valid_pm25,False,0.0003,64,300,5170,3.0956,7.965,3.721,True
4,q3_cfgB_norm.csv,"WD_HR,PM2.5",True,valid_pm25,False,0.0030,32,100,5170,3.0914,7.918,3.756,True
5,q3_cfgB_nonorm.csv,"WD_HR,PM2.5",False,valid_pm25,False,0.0030,32,100,5170,3.3172,6.891,3.721,True
6,q1_row1b_6feats_filter30.csv,"CO,O3,PM10,RH,WIND_SPEED,PM2.5",True,valid_pm25,False,0.0003,64,300,5170,2.9379,8.182,3.913,True
7,q1_row5_PM10_PM25_nonorm.csv,"PM10,PM2.5",False,valid_pm25,False,0.0003,64,300,5170,3.0358,8.045,3.920,True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Round 2 - extra Kaggle experiments (5 submissions left)**

Round 1 (above) filled most of the report. What is still weak or missing, and how the 5 remaining submissions are spent:

| # | csv | report slot | why it makes the answer stronger |
|---|---|---|---|
| 1 | `q3_cfgB_norm.csv` *(already generated in round 1)* | Q3(b) config B, "有 normalization" = **[待填]** | The only blank cell in the report. Completes the B pair (B no-norm is already 3.03863). |
| 2 | `q3_cfgC_norm.csv` *(new)* | Q3(b) config C | Config C = the TA's original sample `train_config`. The A/B Kaggle gaps are only ~0.007 (inside public-LB noise, 45 rows), so the report's "normalisation lets GD converge" argument currently rests on *local* numbers only. C is the setting where the local gap is huge (7.3 vs 54.1), so it can show the effect on Kaggle itself. |
| 3 | `q3_cfgC_nonorm.csv` *(new)* | Q3(b) config C | Pair of #2. |
| 4 | `q1_row1b_6feats_filter30.csv` *(already generated in round 1)* | Q1 table, new row 1b | Report claim (i) "6 features -> 1-2 features is the biggest gain" is confounded: row 1 changes the features **and** the filter (target > 40 vs > 30) at once. Row 1b = 6 features with the *same* <=30 filter as rows 2-4, so row 1b vs row 4 isolates features, row 1b vs row 1 isolates the filter. |
| 5 | `q2_linear_NO_only.csv` *(new)* | Q2(b) | Control for the 4.39283 score: identical NO-only setup, same config / filter / seed, but **linear** instead of 2nd-order. Gives Kaggle evidence (not just local) for the statement that the x^2 terms do not help / over-extrapolate. |

Not spent (lowest value): `q1_row5_PM10_PM25_nonorm.csv` (normalisation is already covered by Q3), a PM10+PM2.5 polynomial model (not what Q2 asks).

Note: the public leaderboard is only 45 of the 90 rows, so differences below ~0.02 RMSE should be described as "no meaningful difference", not as a win.

In [25]:
# ============================================================
# Round 2 experiments  (re-uses run_experiment / valid / valid_pm25 from the helper cell above)
# ============================================================
# Config C = the TA sample config quoted in the report (batch 3, lr 0.1, 5 epochs).
# decay_rate is not stated in the report; 0.9 is assumed (same as every other config in this notebook).
# If the sample code you started from used another decay_rate, change it here BEFORE running.
cfg_C = Namespace(batch_size=3, lr=0.1, epoch=5, decay_rate=0.9)
print('cfg_C =', cfg_C)

feats_wd = [11, 14]     # WD_HR, PM2.5

# --- #2, #3: Q3(b) config C, only `norm` differs (same seed, same valid_pm25 filter) -------------------
r_C_norm   = run_experiment('q3_cfgC_norm',   feats_wd, True,  cfg_C, valid_pm25)
r_C_nonorm = run_experiment('q3_cfgC_nonorm', feats_wd, False, cfg_C, valid_pm25)

# --- #5: Q2 control - the SAME NO-only setup as q2_poly2_NO_only, but linear ----------------------------
# (feats=[2], z-score, train_config, the notebook's original valid(), seed 9487 -> only poly=False differs)
r_lin = run_experiment('q2_linear_NO_only', [2], True, train_config, valid, poly=False)

print()
print('Q2 control, NO-only, parameters (hour 0 = oldest ... 7 = most recent):')
display(pd.DataFrame({'hour': range(8), 'w linear': r_lin['w'],
                      'w1 (x) poly': q2['w'][:8], 'w2 (x^2) poly': q2['w'][8:]}).round(4))
print('bias  linear =', round(r_lin['b'], 4), '| poly =', round(q2['b'], 4))
print('train RMSE  linear =', round(r_lin['train_rmse'], 4), '| poly =', round(q2['train_rmse'], 4),
      ' (poly should be <= linear on train; the Kaggle score tells whether it generalises)')
print('test-pred std  linear =', round(float(np.std(r_lin['pred'])), 3), '| poly =', round(float(np.std(q2['pred'])), 3))
print('max |linear pred - poly pred| =', round(float(np.abs(r_lin['pred'] - q2['pred']).max()), 3))

# --- diagnostics for the report text (no Kaggle submission needed) -------------------------------------
print()
print('Q3 config C - is the no-norm run actually unstable?')
print('  train RMSE  norm =', round(r_C_norm['train_rmse'], 3), '| no-norm =', round(r_C_nonorm['train_rmse'], 3))
print('  max |w|     norm =', round(float(np.abs(r_C_norm['w']).max()), 3), '| no-norm =', round(float(np.abs(r_C_nonorm['w']).max()), 3))
print('  pred std    norm =', round(float(np.std(r_C_norm['pred'])), 3), '| no-norm =', round(float(np.std(r_C_nonorm['pred'])), 3))

cfg_C = Namespace(batch_size=3, lr=0.1, epoch=5, decay_rate=0.9)
q3_cfgC_norm                       n_train= 5170  train_RMSE=   7.9502  pred mean/std=   2.205/   2.181  saved q3_cfgC_norm.csv
q3_cfgC_nonorm                     n_train= 5170  train_RMSE=  43.3876  pred mean/std=  35.323/  10.684  saved q3_cfgC_nonorm.csv
q2_linear_NO_only                  n_train= 5503  train_RMSE=   6.8930  pred mean/std=   6.806/   0.461  saved q2_linear_NO_only.csv

Q2 control, NO-only, parameters (hour 0 = oldest ... 7 = most recent):


,hour,w linear,w1 (x) poly,w2 (x^2) poly
0,0,0.2748,-0.0639,0.0422
1,1,0.3483,-0.0466,0.0682
2,2,0.3567,0.0430,0.0829
3,3,0.2390,0.2173,-0.0036
4,4,0.2271,0.1358,0.0273
5,5,0.1685,-0.3408,0.1042
6,6,0.2615,-0.1883,0.0643
7,7,0.1570,-0.6058,0.1587


bias  linear = 7.6517 | poly = 7.5856
train RMSE  linear = 6.893 | poly = 6.9885  (poly should be <= linear on train; the Kaggle score tells whether it generalises)
test-pred std  linear = 0.461 | poly = 1.146
max |linear pred - poly pred| = 9.195

Q3 config C - is the no-norm run actually unstable?
  train RMSE  norm = 7.95 | no-norm = 43.388
  max |w|     norm = 0.579 | no-norm = 0.653
  pred std    norm = 2.181 | no-norm = 10.684


## **Round 2 manifest - what to upload (5 files, in this order) and where each score goes**

Fill in the Kaggle public scores in the next cell after uploading; it prints the rows ready to paste into the report.

In [26]:
# Upload these 5 csv files (round-2 budget). Two of them were already created by round 1.
upload_plan = [
    ('q3_cfgB_norm.csv',             'Q3(b) row B, "有 normalization"  (fills the [待填] cell)'),
    ('q3_cfgC_nonorm.csv',           'Q3(b) new row C, "無 normalization"'),
    ('q3_cfgC_norm.csv',             'Q3(b) new row C, "有 normalization"'),
    ('q1_row1b_6feats_filter30.csv', 'Q1 table, new row 1b (6 features, SAME <=30 filter)'),
    ('q2_linear_NO_only.csv',        'Q2(b) control: linear NO-only'),
]
missing = [f for f, _ in upload_plan if not os.path.exists(f)]
print('missing files (re-run the round-1 cells for these):', missing if missing else 'none')
for i, (f, slot) in enumerate(upload_plan, 1):
    print(f'{i}. {f:32s} -> {slot}')

# Paste the Kaggle public scores here after uploading (leave None until you have them), then re-run this cell.
kaggle_public = {
    'q3_cfgB_norm.csv': None,
    'q3_cfgC_nonorm.csv': None,
    'q3_cfgC_norm.csv': None,
    'q1_row1b_6feats_filter30.csv': None,
    'q2_linear_NO_only.csv': None,
}
# Already in the report (for the side-by-side comparison):
known = {'Q1 row 4 (PM10+PM2.5, <=30 filter)': 3.05207, 'Q1 row 2 (PM2.5 only, <=30 filter)': 3.13460,
         'Q1 row 1 (6 feats, target>40 filter)': 3.53749,
         'Q2 poly2 NO-only': 4.39283, 'Q3 A norm / no-norm': (3.03433, 3.04134), 'Q3 B no-norm': 3.03863}
if all(v is not None for v in kaggle_public.values()):
    s = kaggle_public
    print()
    print('Q3(b) config B   norm = %.5f | no-norm = 3.03863' % s['q3_cfgB_norm.csv'])
    print('Q3(b) config C   norm = %.5f | no-norm = %.5f   (local only before: 7.308 vs 54.078)' % (s['q3_cfgC_norm.csv'], s['q3_cfgC_nonorm.csv']))
    print('Q1 row 1b (6 feats, <=30)      = %.5f  vs row 4 (PM10+PM2.5, <=30) = 3.05207 -> effect of FEATURES; vs row 1 (6 feats, >40) = 3.53749 -> effect of FILTER' % s['q1_row1b_6feats_filter30.csv'])
    print('Q2 linear NO-only = %.5f  vs poly2 NO-only = 4.39283 -> effect of the x^2 terms' % s['q2_linear_NO_only.csv'])
else:
    print('\n(scores not filled in yet)')

# Download only the round-2 files
try:
    from google.colab import files
    for f, _ in upload_plan:
        files.download(f)
except ImportError:
    print('Not on Colab - csv files are in', os.getcwd())

missing files (re-run the round-1 cells for these): none
1. q3_cfgB_norm.csv                 -> Q3(b) row B, "有 normalization"  (fills the [待填] cell)
2. q3_cfgC_nonorm.csv               -> Q3(b) new row C, "無 normalization"
3. q3_cfgC_norm.csv                 -> Q3(b) new row C, "有 normalization"
4. q1_row1b_6feats_filter30.csv     -> Q1 table, new row 1b (6 features, SAME <=30 filter)
5. q2_linear_NO_only.csv            -> Q2(b) control: linear NO-only

(scores not filled in yet)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>